In [22]:
2+2

4

In [1]:
import os
import time
import re
from dotenv import load_dotenv


from langchain_core.documents import Document
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda





C:\Users\vaish\AppData\Local\Temp\ipykernel_15376\696281168.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [2]:
# 2.secure authentication
load_dotenv()

api_key=os.getenv("GEMINI_API_KEY")

llm=ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0.0
)

embedder=GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

In [3]:
#3 DATA INGESTION
audit_data = [

Document(page_content="MLOps Audit Q4: European division legacy branches report a 15% OCR failure rate.", metadata={"id": 1}),
Document(page_content="Tuesday Review confirmed the 15% spike is due to 'Legacy Scan-X' hardware and firmware v2.1.", metadata={"id": 2}),
Document(page_content="Jaymin approved a $45,000 emergency budget to upgrade European scanners by Q1 end.", metadata={"id": 3}),
Document(page_content="OCR failures peak on Tuesdays due to weekly bulk-batch processing of handwritten PDFs.", metadata={"id": 4}),
Document(page_content="Tony recommends a distributed architecture for handling 500+ PDFs in legacy branches.", metadata={"id": 5}),
Document(page_content="The 15% error rate is classified as 'Critical' for Banking and Compliance audits.", metadata={"id": 6}),
Document(page_content="Marten's team is monitoring OCR logs 24/7 until the hardware upgrade is finished.", metadata={"id": 7}),
Document(page_content="European legacy branches are the only units still using the v2.1 firmware.", metadata={"id": 8})

]



In [4]:
vectorstore=FAISS.from_documents(audit_data,embedder)

base_retriever=vectorstore.as_retriever(search_kwargs={"k":3})

In [5]:
#4.PROMPT (the brains)

#A. STEP BACK PROMPT

step_back_prompt = ChatPromptTemplate.from_template(
"""You are an expert at world knowledge.
Paraphrase the following question into a SINGLE, concise, foundational step-back question.
Output ONLY the question and nothing else.

Original Question: {question}
Step-Back Question:"""

)

step_back_chain=step_back_prompt|llm|StrOutputParser()

In [6]:
# B. Router Prompt

router_prompt = ChatPromptTemplate.from_template(

"""You are a query router. Classify the user's question into one of two categories:
- 'CALCULATE' (if it requires math, budget calculation, or counting)
- 'SEARCH' (if it requires looking up facts, hardware issues, or audit logs)


Output ONLY the exact category word.
Question: {question}
Category:"""

)

router_chain = router_prompt | llm | StrOutputParser()

In [7]:
# C. Decomposition Prompt

decomp_prompt = ChatPromptTemplate.from_template(

"""You are a helpful assistant. Break the following complex question into 3 simpler, independent sub-questions.
Output them on separate lines without numbers or bullets.
Original: {question}"""

)

decomp_chain = decomp_prompt | llm | StrOutputParser()

In [8]:
# D. HyDE Prompt

hyde_prompt = ChatPromptTemplate.from_template(

"""You are an expert MLOps auditor. Write a plausible, hypothetical audit log entry that would perfectly answer this question.
Do not use introductory filler. Just write the hypothetical log.
Question: {question}

Hypothetical Log:"""

)

hyde_chain = hyde_prompt | llm | StrOutputParser()

In [13]:
# E. Final Generation Prompt

response_prompt = ChatPromptTemplate.from_template(

"""You are an elite MLOps auditor. Answer using ONLY this context.

Foundational Context: {step_back_context}
Specific Context: {normal_context}
Question: {question}
Final Answer:"""

)

In [9]:
#5 Retrieval stratergies


def run_stepback_rag(query:str):
    raw_broad_quest=step_back_chain.invoke({"question":query})
    #string manipulation
    lines=raw_broad_quest.split('\n')
    broad_q=next((line.strip() for line in lines if "?" in line),lines[0].strip()) #strip ? if not fallback to lines[0]-only one question is generated
    print(f"---->cleaned broad query:{broad_q}")
    n_docs=base_retriever.invoke(query)  #retrieved context based on user query
    time.sleep(2.5) #api cool down period prevent rate limit
    sb_docs=base_retriever.invoke(broad_q) #retrieved context based on step back question
    return{

        "step_back_context":"\n".join([doc.page_content for doc in sb_docs]),
        "normal_context":"\n".join([doc.page_content for doc in n_docs]),
        "question":query
    }




In [18]:
def run_multiquery_rag(query: str):

        print("\n[Strategy: Multi-Query] Shattering the Hydra...")
        sub_queries_raw = decomp_chain.invoke({"question": query})
        sub_queries = [q.strip() for q in sub_queries_raw.split('\n') if q.strip()]
        pooled_docs = []

        for sq in sub_queries:
            print(f" -> Sub-query: {sq}")
            docs = base_retriever.invoke(sq)
            pooled_docs.extend(docs)
            time.sleep(2.5)
            unique_docs = {doc.metadata['id']: doc for doc in pooled_docs}.values()

            return {

            "step_back_context": "N/A for Multi-Query",
            "normal_context": "\n\n".join(d.page_content for d in unique_docs),
            "question": query

            }

In [26]:
def run_hyde_rag(query: str):

    print("\n[Strategy: HyDE] Generating hypothetical document...")
    hypothetical_doc = hyde_chain.invoke({"question": query})
    print(f" -> Fake Doc: {hypothetical_doc[:100]}...\n")

    retrieved_docs = base_retriever.invoke(hypothetical_doc)


    return {

    "step_back_context": "HyDE Retrieval Used",
    "normal_context": "\n\n".join(d.page_content for d in retrieved_docs),
    "question": query

    }

In [10]:
def route_query(query: str, selected_strategy):

    print(f"\n[Traffic Cop] Analyzing intent...")
    category = router_chain.invoke({"question": query}).strip()
    print(f" -> Routed to: {category}")


    if category == "CALCULATE":
            return {"error": "Route to Math Tool (Skipping Vector DB to save costs/hallucinations)."}
    else:
            return selected_strategy(query)

In [32]:
# 6. Assemble and run

ACTIVE_STRATEGY = run_stepback_rag # Options: run_stepback_rag, run_multiquery_rag, run_hyde_rag

test_queries = [

"Why did the OCR failure rate specifically spike to 15% for the legacy branch offices in the European division during the Tuesday review?",
"Who approved the budget, what hardware is failing, and who is monitoring it?",
"What is the total emergency budget approved for the upgrade multiplied by 2?",
"who is prime minister of india and which party he belongs"

]

current_query = test_queries[0]

In [33]:
try:

    print(f"\n--- INITIATING DEEP AUDIT FOR: '{current_query}' ---")
    retrieval_output = route_query(current_query, ACTIVE_STRATEGY)
    print("**************"*40)
    print(retrieval_output)

    if "error" in retrieval_output:
         print(f"\nExecution Halted: {retrieval_output['error']}")

    else:
        final_answer = (response_prompt | llm | StrOutputParser()).invoke(retrieval_output)
        print(f"\n[Final Answer]:\n{final_answer}")

except Exception as e:
    print(f"\n[Fatal Error]: {e}")


--- INITIATING DEEP AUDIT FOR: 'Why did the OCR failure rate specifically spike to 15% for the legacy branch offices in the European division during the Tuesday review?' ---

[Traffic Cop] Analyzing intent...


c:\Users\vaish\agenticaisep\.venv13\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


 -> Routed to: SEARCH


c:\Users\vaish\agenticaisep\.venv13\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


---->cleaned broad query:What caused the OCR failure rate to increase in the European division?
********************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************************
{'step_back_context': "MLOps Audit Q4: European division legacy branches report a 15% OCR failure rate.\nOCR failures peak on Tuesdays due to weekly bulk-batch processing of handwritten PDFs.\nMarten's team is monitoring OCR logs 24/7 until the hardware upgrade is finished.", 'normal_context': "MLOps Audit Q4: European division legacy branch

c:\Users\vaish\agenticaisep\.venv13\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



[Final Answer]:
Based on the provided context, the OCR failure rate specifically spiked to 15% during the Tuesday review due to 'Legacy Scan-X' hardware and firmware v2.1.
